In [1]:
import sys
import os
import json
import torch
import faiss
import pymupdf
import ollama
import numpy as np
from tqdm import tqdm


from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, logging
import transformers.utils as transformers_utils

# niech się tylko wyświetlają błędy
logging.set_verbosity_error()

model_id = "qwen3:4b"

# Model embeddingowy – zamienia tekst na wektory liczbowe (embeddingi)
# all-mpnet-base-v2 produkuje wektory o wymiarze 768
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Tokenizer i model obsługiwane przez Ollama
tokenizer = None
model = None


In [2]:
# Tworzymy pusty indeks FAISS typu FlatL2 – przechowuje wektory i szuka po odległości euklidesowej
# get_embedding_dimension() zwraca wymiar wektorów modelu embeddingowego (768)
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())

# Lista słowników przechowująca metadane każdego chunka (nazwa pliku, numer strony, tekst)
metadata = []

print('Number of chunks: ', index.ntotal) # 0

Number of chunks:  0


In [3]:
class Utils:
    def __init__(self, embedding_model: SentenceTransformer=None, llm_model: AutoModelForCausalLM=None, llm_tokenizer: AutoTokenizer=None, index=None, metadata=None, chunk_size=512):
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.llm_tokenizer = llm_tokenizer
        self.index = index
        self.metadata = metadata
        # Rozmiar chunka w znakach – określa jak długie będą fragmenty tekstu
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        """
        Extract text from PDF file. Returns a list of tuples (page_number, text).
        """
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Zamieniamy znaki nowej linii na spacje, żeby tekst był ciągły
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            # Dzielimy tekst strony na fragmenty o długości chunk_size znaków
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            # Zamieniamy tekst chunka na wektor embeddingowy
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            # Dodajemy wektor do indeksu FAISS
            self.index.add(np.array([embeddings]))
            # Zapisujemy metadane chunka – powiążemy je z wektorem przez pozycję w indeksie
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        # Zapisujemy indeks FAISS na dysk, żeby nie trzeba było go odbudowywać przy każdym uruchomieniu
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """
        Process the file and add chunks to FAISS index
        """
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0

        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)
    
    def answer_question(self, prompt_template="", query="", max_tokens=512, temp=0.7, k=5):
        # Zamieniamy pytanie użytkownika na wektor embeddingowy
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        # Szukamy k najbliższych wektorów w FAISS – D to odległości, I to indeksy znalezionych chunków
        D, I = self.index.search(np.array([question_embedding]), k)
        # Pobieramy metadane (tekst) znalezionych chunków
        chunks = [self.metadata[i] for i in I[0]]

        # Sklejamy teksty chunków w jeden blok kontekstu dla modelu
        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        # Wstawiamy kontekst i pytanie do szablonu promptu
        prompt = prompt_template.format(context=context, query=query)

        # Budujemy historię rozmowy w formacie wymaganym przez modele czatu
        messages = [
            {
                "role": "system",
                "content": (
                    "Be helpful, straight to the point. "
                    "Use only context. Do not hallucinate."
                )
            },
            {"role": "user", "content": prompt},
        ]

        # Wysyłamy zapytanie do lokalnego modelu przez bibliotekę ollama
        response = ollama.chat(
            model=model_id,
            messages=messages,
            options={
                "temperature": temp,
                "num_predict": max_tokens,
            }
        )
        answer = response.message.content

        return answer, chunks

In [4]:
# Tworzymy obiekt Utils łącząc wszystkie komponenty RAG w jednym miejscu
utils = Utils(
    embedder,   # model embeddingowy do zamiany tekstu na wektory
    model,      # LLM (None – obsługiwany przez Ollama)
    tokenizer,  # tokenizer (None – obsługiwany przez Ollama)
    index,      # indeks FAISS z wektorami chunków
    metadata,   # metadane chunków (tekst, strona, plik)
    chunk_size=512 # ilość treści w każdym fragmencie - zwiększamy by dać modelowi więcej treści gdy odmówi odpowiedzi
)

In [5]:
knowledge_dir = "my_knowledge/"
# Przetwarzamy każdy plik PDF z katalogu – dzielimy na chunki i dodajemy do FAISS
for file in os.listdir(knowledge_dir):
    utils.process_file(knowledge_dir + file)
    
print('Number of chunks: ', index.ntotal)

Adding chunks to FAISS:   0%|          | 0/27 [00:00<?, ?it/s]c:\Users\Sebastian\Studia\Sem_6\SI\lab7\.venv\Lib\site-packages\transformers\models\xlm_roberta\modeling_xlm_roberta.py:364: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Adding chunks to FAISS: 100%|██████████| 170/170 [00:04<00:00, 41.95it/s]
Adding chunks to FAISS: 0it [00:00, ?it/s]
Adding chunks to FAISS: 0it [00:00, ?it/s]

Number of chunks:  351


In [6]:
# przykładowe pytania dotyczące dokumentów o fokach
questions = [
    "Jak wygląda foka pospolita (harbour seal) i gdzie występuje?",
    "Jakie jest typowe zachowanie rozrodcze fok pospolitych?",
    "Co jedzą foki pospolite i jak polują?",
    "Jak długo żyją foki pospolite i kiedy osiągają dojrzałość płciową?",
    "Jakie są główne zagrożenia dla populacji fok pospolitych?",
    "Jak zmieniały się liczebności fok pospolitych w ostatnich latach?",
    "Jakie choroby i pasożyty atakują foki pospolite?",
    "Co wiemy o interakcjach między ludźmi a foką lamparcią (leopard seal)?",
    "Czy foka lamparcią jest niebezpieczna dla ludzi?",
    "W jaki sposób nurkowie i badacze kontaktują się z fokami lamparciami?",
    "Jak zachowuje się foka lamparcią podczas spotkania z człowiekiem pod wodą?",
    "Jak zmieniały się populacje fok w Morzu Bałtyckim według wskaźników HELCOM?",
    "Które gatunki fok zamieszkują Morze Bałtyckie?",
    "Jakie są trendy populacyjne foki szarej i foki obrączkowanej w Bałtyku?",
    "Czym różnią się kotiki (fur seals) od fok pospolitych?",
    "Gdzie zamieszkują kotiki i jak są rozmieszczone na świecie?",
    "Jakie metody stosuje się do schwytania fok pospolitych w celu badań naukowych?",
    "W jaki sposób oznakować fokę tak, by nie zranić zwierzęcia?",
    "Jakie rodzaje znaczników (tagów) stosuje się przy foce pospolitej?",
    "Jak foki są unieruchamiane podczas badań?",
    "Jakie środki ostrożności obowiązują przy pracy z fokami w terenie?",
    "Jak ocenia się stan zdrowia foki podczas schwytania?",
    "Jakie dane zbierane są przy pomiarach biometrycznych fok?",
]

In [7]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=4096,
        temp=0.1
)

display(Markdown(f"#**Pytanie:** {random_query}"))
display(Markdown(f"##**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

#**Pytanie:** Jak zmieniały się liczebności fok pospolitych w ostatnich latach?

##**Odpowiedź:**

W ostatnich latach liczebność populacji fok pospolitych w Wadden Sea zaczęła spadać w ciągu ostatnich pięciu lat. Przed tym okresie (2003–2016) populacja wzrosła rocznie o średnio 7,9%. W 2024 r. zliczono 8 230 młodych fok i 23 772 fok moulting. Brak masowych zgonów w ostatnich latach, ale zmiana trendu może wynikać z konkurencji z innymi gatunkami.

---
**Źródła:**

**[1]** `Population-trends-and-abundance-of-seals-HELCOM-core-indicator-2018.pdf` — strona 10

> bottle-neck in the 1970s when only some 30  seals were counted. Long-term isolation and low numbers have resulted in low genetic variation in this  population (Härkönen et al. 2006). The population h...

**[2]** `Harbour_Seal_Report_2023.pdf` — strona 2

> ther conditions  and disturbance of the seals, may influence the local counts. This is why trends  in abundance should be considered over several years. On longer terms, habitat  alterations such as c...

**[3]** `Harbour_Seal_Report_2024.pdf` — strona 4

> ights are needed to ascertain when  the peaks in pupping and moulting  occur and to determine the proportion  of seals on land during the surveys.   CONCLUSION In 2024, 8,230 harbour seal pups and  23...

**[4]** `Harbour_Seal_Report_2024.pdf` — strona 3

> nually since 2020.  The percentage of pups as related  to the moult count has generally  (see ICES 2024) and there have not  been mass mortality events in recent  years. Neither the number of dead  ha...

**[5]** `Population-trends-and-abundance-of-seals-HELCOM-core-indicator-2018.pdf` — strona 11

> traits and the Southern Baltic Sea   Harbour seals in this area experienced a mass mortality caused by a Phocine Distemper Virus (PDV) epidemic  in 2002 which is why the growth rate is analyzed over a...